# PySpark Interview Practice Notebook

**56 carefully selected problems** covering the most frequently asked topics in Data Engineer interviews.

### How to use this notebook
1. Run the Setup cell first.
2. Read the problem carefully.
3. Try to solve it **without looking at the hint**.
4. Only expand the hint after struggling for a few minutes.
5. Write your solution in the empty code cell below each problem.

**Pro tip**: After solving, always ask yourself:
- Does this cause a shuffle?
- Would this scale to 1TB data?
- Is there a better (more Spark-native) way?

## 0. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("PySpark-Interview-Practice") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready!")
print(f"Spark version: {spark.version}")

---
# Section 1: Basics & Spark Concepts (Problems 1-8)

### Problem 1: Create a DataFrame from a Python list
Create a DataFrame with columns `name` (string) and `age` (integer) from the following data:
```python
data = [("Alice", 28), ("Bob", 34), ("Charlie", 25)]
```
**Hint**: Use `spark.createDataFrame()` or `spark.createDataFrame(data, schema)`.

In [ ]:
# Your solution here


### Problem 2: Difference between Transformation and Action
Explain (in comments or print statements) what happens when you run the following code.
Is `df.filter(...)` executed immediately? Why or why not?

```python
df = spark.range(1000000)
filtered = df.filter(F.col("id") % 2 == 0)
# What happens here?
```
**Hint**: Think about lazy evaluation and the DAG.

In [ ]:
# Your explanation / code here


### Problem 3: Narrow vs Wide Transformation
Classify the following operations as **narrow** or **wide**:
1. `df.filter(F.col("age") > 30)`
2. `df.groupBy("dept").count()`
3. `df.select("name", "age")`
4. `df.join(df2, "id")`
5. `df.withColumn("new_col", F.col("age") * 2)`

**Hint**: Wide transformations require a shuffle (data movement across partitions).

In [ ]:
# Write your classification


### Problem 4: Create DataFrame with explicit Schema
Create a DataFrame using an explicit StructType schema for the data:
```python
data = [("Alice", 28, 50000.0), ("Bob", 34, 60000.5)]
```
Columns: name (String), age (Integer), salary (Double)

**Hint**: Use `StructType([StructField(...), ...])`

In [ ]:
# Your solution here


### Problem 5: Read a CSV (simulate)
Assume you have a CSV file. Write the code to read it with header=True and inferSchema=True.
Also write the version where you provide an explicit schema (best practice for production).

**Hint**: `spark.read.csv(...)` and `.schema(your_schema)`

In [ ]:
# Your solution here


### Problem 6: RDD vs DataFrame – When to use which?
Write a short explanation (3-5 lines) of when you would still use RDDs instead of DataFrames in 2025+.

**Hint**: Think about unstructured data, very custom partitioning, or legacy code.

In [ ]:
# Your answer here


### Problem 7: Actions that trigger computation
List at least 6 common actions in PySpark.

**Hint**: Anything that returns a value to the driver or writes to storage.

In [ ]:
# Your list here


### Problem 8: SparkSession configuration
Write code to create a SparkSession that:
- Has app name "MyApp"
- Uses local mode with all cores
- Sets shuffle partitions to 4
- Enables Adaptive Query Execution

**Hint**: `.config("spark.sql.adaptive.enabled", "true")`

In [ ]:
# Your solution here


---
# Section 2: DataFrame Operations (Problems 9-18)

In [ ]:
# Sample data for many problems in this section
employees_data = [
    (1, "Alice", "Engineering", 28, 75000, None),
    (2, "Bob", "Engineering", 34, 82000, "alice@email.com"),
    (3, "Charlie", "HR", 29, 65000, None),
    (4, "Diana", "HR", 31, 68000, "diana@email.com"),
    (5, "Eve", "Marketing", 26, 62000, None),
    (6, "Frank", "Engineering", 40, 95000, "frank@email.com"),
    (7, "Grace", "Marketing", 33, 71000, None),
    (8, "Alice", "Engineering", 28, 75000, None),  # duplicate
]

emp_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("email", StringType(), True),
])

employees = spark.createDataFrame(employees_data, emp_schema)
employees.show()

### Problem 9: Select and Rename columns
From `employees`, select only `name` and `salary`, and rename `salary` to `annual_salary`.

**Hint**: `.select()` + `.withColumnRenamed()` or `alias()`.

In [ ]:
# Your solution here


### Problem 10: Filter rows
Filter employees who are in Engineering and have salary > 80000.

**Hint**: Use `&` for AND (remember parentheses).

In [ ]:
# Your solution here


### Problem 11: Handle Nulls – Drop
Drop all rows where `email` is null.

**Hint**: `df.na.drop(subset=["email"])` or `df.dropna(subset=["email"])`

In [ ]:
# Your solution here


### Problem 12: Handle Nulls – Fill
Fill null emails with the string "unknown@company.com".

**Hint**: `df.na.fill({"email": "unknown@company.com"})`

In [ ]:
# Your solution here


### Problem 13: Remove exact duplicates
Remove exact duplicate rows from `employees`.

**Hint**: `.dropDuplicates()` or `.distinct()`

In [ ]:
# Your solution here


### Problem 14: Add a new column
Add a column `salary_band`:
- "High" if salary >= 80000
- "Medium" if 65000 <= salary < 80000
- "Low" otherwise

**Hint**: Use `F.when().when().otherwise()`

In [ ]:
# Your solution here


### Problem 15: Cast column type
Cast the `salary` column to DoubleType and show the schema.

**Hint**: `.withColumn("salary", F.col("salary").cast("double"))`

In [ ]:
# Your solution here


### Problem 16: Sort / OrderBy
Sort employees by `dept` ascending and `salary` descending.

**Hint**: `.orderBy(F.col("dept").asc(), F.col("salary").desc())`

In [ ]:
# Your solution here


### Problem 17: Distinct values of a column
Get the distinct department names.

**Hint**: `.select("dept").distinct()`

In [ ]:
# Your solution here


### Problem 18: Count nulls per column
Write code that shows the number of nulls in each column of `employees`.

**Hint**: Use a list comprehension with `F.count(F.when(F.col(c).isNull(), c))`

In [ ]:
# Your solution here


---
# Section 3: Aggregations & GroupBy (Problems 19-24)

### Problem 19: Basic groupBy + agg
For each department, calculate:
- number of employees
- average salary
- max salary

**Hint**: `.groupBy("dept").agg(F.count("*"), F.avg("salary"), F.max("salary"))`

In [ ]:
# Your solution here


### Problem 20: Multiple aggregations with aliases
Same as above but give nice column names: `emp_count`, `avg_salary`, `max_salary`.

**Hint**: Use `.alias("name")` inside agg.

In [ ]:
# Your solution here


### Problem 21: Filter after aggregation (HAVING equivalent)
Show only departments that have more than 2 employees.

**Hint**: Aggregate first, then `.filter()` on the aggregated column.

In [ ]:
# Your solution here


### Problem 22: collect_list / collect_set
For each department, create a list of employee names.

**Hint**: `F.collect_list("name")` or `F.collect_set("name")`

In [ ]:
# Your solution here


### Problem 23: Pivot
Create a pivot table showing average salary by department (rows) and ... wait, we only have one dimension.
Instead: Create sample data with year and show average salary pivoted by year.

**Hint**: `.groupBy("dept").pivot("year").agg(F.avg("salary"))`

In [ ]:
# First create a small dataset with year, then pivot


### Problem 24: Count distinct
How many distinct departments are there? How many distinct names?

**Hint**: `F.countDistinct("dept")`

In [ ]:
# Your solution here


---
# Section 4: Joins (Problems 25-31)

In [ ]:
# Sample data for joins
dept_data = [
    ("Engineering", "Building A"),
    ("HR", "Building B"),
    ("Finance", "Building C"),  # no employees
]
departments = spark.createDataFrame(dept_data, ["dept_name", "location"])

# employees already exists from earlier
employees.show()
departments.show()

### Problem 25: Inner Join
Join `employees` and `departments` on department name (inner join).

**Hint**: `employees.join(departments, employees.dept == departments.dept_name, "inner")`

In [ ]:
# Your solution here


### Problem 26: Left Join
Do a left join so that all employees are kept even if their department is missing in the departments table.

**Hint**: `how="left"`

In [ ]:
# Your solution here


### Problem 27: Left Anti Join
Find employees whose department does **not** exist in the departments table.

**Hint**: `how="left_anti"`

In [ ]:
# Your solution here


### Problem 28: Left Semi Join
Find employees whose department **does** exist in the departments table (without bringing department columns).

**Hint**: `how="left_semi"`

In [ ]:
# Your solution here


### Problem 29: Broadcast Join
Force a broadcast join of the small `departments` table.

**Hint**: `from pyspark.sql.functions import broadcast` then `employees.join(broadcast(departments), ...)`

In [ ]:
# Your solution here


### Problem 30: Self Join
Find pairs of employees who work in the same department (exclude pairing a person with themselves).

**Hint**: Alias the DataFrame twice and join on dept + id != id.

In [ ]:
# Your solution here


### Problem 31: Join with different column names
Join when the key columns have different names.

**Hint**: Use the expression form: `df1.join(df2, df1.colA == df2.colB, "inner")`

In [ ]:
# Your solution here


---
# Section 5: Window Functions (Problems 32-41)

This is one of the **most important** sections for interviews.

In [ ]:
# Richer sample data for window problems
sales_data = [
    (1, "Alice", "2024-01-05", 100),
    (2, "Bob", "2024-01-07", 150),
    (3, "Alice", "2024-01-10", 200),
    (4, "Charlie", "2024-01-12", 80),
    (5, "Bob", "2024-01-15", 120),
    (6, "Alice", "2024-01-20", 300),
    (7, "Charlie", "2024-01-22", 90),
    (8, "Bob", "2024-01-25", 180),
    (9, "Alice", "2024-02-01", 250),
    (10, "Charlie", "2024-02-03", 110),
]

sales = spark.createDataFrame(sales_data, ["id", "salesperson", "sale_date", "amount"])
sales = sales.withColumn("sale_date", F.to_date("sale_date"))
sales.show()

### Problem 32: row_number
Add a column that ranks each sale within the salesperson by amount descending (1 = highest sale).

**Hint**: `Window.partitionBy("salesperson").orderBy(F.col("amount").desc())` + `F.row_number()`

In [ ]:
# Your solution here


### Problem 33: rank vs dense_rank
Add both `rank` and `dense_rank` columns ordered by amount descending (across all data).
Explain the difference when there are ties.

**Hint**: Create a small DataFrame with duplicate amounts to demonstrate.

In [ ]:
# Your solution here


### Problem 34: Top-N per group
Get the top 2 highest sales for each salesperson.

**Hint**: Use row_number() and then filter `rn <= 2`.

In [ ]:
# Your solution here


### Problem 35: Running total / Cumulative sum
For each salesperson, calculate the running total of sales ordered by date.

**Hint**: `F.sum("amount").over(Window.partitionBy("salesperson").orderBy("sale_date").rowsBetween(Window.unboundedPreceding, 0))`

In [ ]:
# Your solution here


### Problem 36: Lag and Lead
For each salesperson, add columns showing the previous sale amount and the next sale amount.

**Hint**: `F.lag("amount", 1)` and `F.lead("amount", 1)` over a window ordered by date.

In [ ]:
# Your solution here


### Problem 37: Percentage of total
For each sale, calculate what percentage it represents of the salesperson's total sales.

**Hint**: Use a window without orderBy to get the total, then divide.

In [ ]:
# Your solution here


### Problem 38: Moving Average
Calculate a 3-sale moving average of amount for each salesperson (ordered by date).

**Hint**: `rowsBetween(-2, 0)`

In [ ]:
# Your solution here


### Problem 39: Difference from previous
Calculate the difference in amount compared to the previous sale for the same salesperson.

**Hint**: `F.col("amount") - F.lag("amount", 1).over(...)`

In [ ]:
# Your solution here


### Problem 40: First and Last value
For each salesperson, add columns showing their first sale amount and last sale amount (by date).

**Hint**: `F.first("amount").over(...)` and `F.last("amount").over(...)` with proper frame.

In [ ]:
# Your solution here


### Problem 41: Deduplicate keeping latest record (classic interview question)
Given a DataFrame of customer updates, keep only the latest record per customer_id based on updated_at.

**Hint**: `row_number()` over partitionBy customer_id orderBy updated_at desc, then filter rn == 1.

In [ ]:
# Create sample data and solve
updates = spark.createDataFrame([
    (101, "Alice", "2024-01-01", 100),
    (101, "Alice", "2024-01-15", 150),
    (102, "Bob", "2024-01-05", 200),
    (101, "Alice", "2024-02-01", 180),
    (102, "Bob", "2024-01-20", 220),
], ["customer_id", "name", "updated_at", "score"])

# Your solution here


---
# Section 6: Performance & Optimization Concepts (Problems 42-46)

### Problem 42: repartition vs coalesce
Explain the difference and when you would use each.
Write code examples of both.

**Hint**: repartition does full shuffle, coalesce avoids full shuffle when decreasing partitions.

In [ ]:
# Your explanation + code


### Problem 43: Caching
When should you cache a DataFrame? Show the correct way to cache and unpersist.

**Hint**: Only when the DataFrame is used multiple times in the same job.

In [ ]:
# Your solution here


### Problem 44: Broadcast Join decision
You have a 500GB fact table and a 50MB dimension table. What join strategy should you use and why?
Write the code.

**Hint**: Broadcast the small table.

In [ ]:
# Your answer + code pattern


### Problem 45: Data Skew
What is data skew? Name three techniques to handle it in PySpark.

**Hint**: Salting, Broadcast, AQE skew join handling.

In [ ]:
# Your answer


### Problem 46: Explain plan
How do you inspect the physical plan of a DataFrame? Why is it useful?

**Hint**: `df.explain()` or `df.explain(True)` / `df.explain("formatted")`

In [ ]:
# Your solution here


---
# Section 7: Real Coding Scenarios (Problems 47-56)

### Problem 47: Second highest salary per department
Using the employees DataFrame, find the second highest salary in each department.

**Hint**: dense_rank() or rank() and filter rank == 2.

In [ ]:
# Your solution here


### Problem 48: Employees who earn more than their department average
Find all employees whose salary is greater than the average salary of their department.

**Hint**: Window with avg, or self-join with aggregated averages.

In [ ]:
# Your solution here


### Problem 49: Consecutive days with sales
Find salespersons who had sales on 3 consecutive days.
(You may need to create suitable sample data).

**Hint**: Use lag to get previous dates and check date differences.

In [ ]:
# Your solution here


### Problem 50: Explode an array column
Given a DataFrame with a column of arrays, explode it into multiple rows.

**Hint**: `F.explode("array_col")`

In [ ]:
# Create sample + solve
data = [("Alice", ["Python", "Spark"]), ("Bob", ["Java", "SQL", "Airflow"])]
df = spark.createDataFrame(data, ["name", "skills"])

# Your solution here


### Problem 51: Flatten nested struct
Given a DataFrame with a nested struct column, flatten it.

**Hint**: `df.select("id", "address.*")` or use `F.col("address.city")`

In [ ]:
# Create sample + solve


### Problem 52: Word Count (classic)
Implement word count on a text RDD or DataFrame.

**Hint**: flatMap → map to (word,1) → reduceByKey or groupBy + count

In [ ]:
# Your solution here


### Problem 53: Remove duplicate rows but keep the one with max value
For each id, keep the row that has the maximum score.

**Hint**: Window + row_number ordered by score desc.

In [ ]:
# Your solution here


### Problem 54: Calculate median
Calculate the median salary per department.

**Hint**: Use percentile_approx or a window approach.

In [ ]:
# Your solution here


### Problem 55: Sessionization (simple version)
Given user click events with timestamps, assign a session_id where a new session starts if the gap from the previous event is > 30 minutes.

**Hint**: lag + unix_timestamp difference + cumulative sum of new session flags.

In [ ]:
# Advanced – try it!


### Problem 56: Write a reusable function
Write a Python function that takes a DataFrame and a list of key columns and returns the DataFrame with duplicates removed, keeping the latest record based on a timestamp column.

**Hint**: This is the production version of Problem 41.

In [ ]:
def deduplicate_latest(df, key_cols, timestamp_col):
    # Your implementation here
    pass

# Test it


---

## 🎉 Congratulations!

You have completed the main practice set.

### Next Steps
1. Re-solve the Window function problems from memory.
2. Time yourself on Problems 41, 47, 48, 56.
3. Practice explaining your solutions out loud.
4. Look at Spark UI for a few of your jobs (especially joins and window operations).

**Star the repo if this helped you!**

Happy interviewing! 🔥